In [1]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [2]:
from scripts.model import DonutConfig, DonutModel
from scripts.tokenizer import get_tokenizer

W0930 19:18:21.133000 22100 site-packages/torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0930 19:18:21.170000 22100 site-packages/torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


In [3]:
tokenizer = get_tokenizer()

In [4]:
config = DonutConfig(
    input_size=(1280, 960),
    window_size=10,
    encoder_layer=(2, 2, 14, 2),
    max_position_embeddings=768,
    decoder_layer=4,
    vocab_size=len(tokenizer),
    pad_token_id=tokenizer.pad_token_id,
)

In [6]:
model = DonutModel(config)

In [8]:
sum(p.numel() for p in model.parameters())

201109624

In [9]:
sum(p.numel() for p in model.encoder.parameters())

74180728

In [10]:
sum(p.numel() for p in model.decoder.parameters())

126928896

In [11]:
74180728 + 126928896

201109624

### Encoder

In [22]:
import timm
import math
import torch.nn.functional as F

In [23]:
def get_pretrained_encoder(encoder, config):
    swin_state_dict = timm.create_model("swin_base_patch4_window12_384", pretrained=True).state_dict()
    new_swin_state_dict = encoder.state_dict()
    for x in new_swin_state_dict:
        if x.endswith("relative_position_index") or x.endswith("attn_mask"):
            pass
        elif (
            x.endswith("relative_position_bias_table")
            and encoder.layers[0].blocks[0].attn.window_size[0] != 12
        ):
            pos_bias = swin_state_dict[x].unsqueeze(0)[0]
            old_len = int(math.sqrt(len(pos_bias)))
            new_len = int(2 * config.window_size - 1)
            pos_bias = pos_bias.reshape(1, old_len, old_len, -1).permute(0, 3, 1, 2)
            pos_bias = F.interpolate(pos_bias, size=(new_len, new_len), mode="bicubic", align_corners=False)
            new_swin_state_dict[x] = pos_bias.permute(0, 2, 3, 1).reshape(1, new_len ** 2, -1).squeeze(0)
        else:
            new_swin_state_dict[x] = swin_state_dict[x]   
    return new_swin_state_dict

In [24]:
encoder_state_dict = get_pretrained_encoder(model.encoder.model, config)

In [26]:
model.encoder.model.load_state_dict(encoder_state_dict)

<All keys matched successfully>

### Decoder

In [29]:
from transformers import MBartConfig, MBartForCausalLM, XLMRobertaTokenizer
import torch

In [33]:
def get_pretrained_decoder(decoder, config, pad_token_id):
    def resize_bart_abs_pos_emb(weight: torch.Tensor, max_length: int) -> torch.Tensor:
        if weight.shape[0] > max_length:
            weight = weight[:max_length, ...]
        else:
            weight = (
                F.interpolate(
                    weight.permute(1, 0).unsqueeze(0),
                    size=max_length,
                    mode="linear",
                    align_corners=False,
                )
                .squeeze(0)
                .permute(1, 0)
            )
        return weight

    _model = MBartForCausalLM(
        config=MBartConfig(
            is_decoder=True,
            is_encoder_decoder=False,
            add_cross_attention=True,
            decoder_layers=config.decoder_layer,
            max_position_embeddings=config.max_position_embeddings,
            vocab_size=config.vocab_size,
            scale_embedding=True,
            add_final_layer_norm=True,
        )
    )

    _model.forward = decoder.forward
    _model.config.is_encoder_decoder = True
    _model.model.decoder.embed_tokens.padding_idx = pad_token_id
    _model.prepare_inputs_for_generation = decoder.prepare_inputs_for_inference

    # Load pretrained BART
    bart_state_dict = MBartForCausalLM.from_pretrained(
        "hyunwoongko/asian-bart-ecjk"
    ).state_dict()

    new_bart_state_dict = _model.state_dict()

    pretrained_vocab_size = bart_state_dict["model.decoder.embed_tokens.weight"].shape[0]
    target_vocab_size = config.vocab_size

    for x in new_bart_state_dict:
        if x.endswith("embed_positions.weight") and config.max_position_embeddings != 1024:
            new_bart_state_dict[x] = resize_bart_abs_pos_emb(
                bart_state_dict[x],
                config.max_position_embeddings + 2,
            )

        elif x.endswith("embed_tokens.weight") or x.endswith("lm_head.weight"):
            pretrained_weight = bart_state_dict[x]

            new_weight = new_bart_state_dict[x].clone()

            # Copy pretrained vocabulary
            copy_size = min(pretrained_weight.shape[0], target_vocab_size)

            new_weight[:copy_size] = pretrained_weight[:copy_size]

            new_bart_state_dict[x] = new_weight

        else:
            new_bart_state_dict[x] = bart_state_dict[x]

    return new_bart_state_dict

In [34]:
decoder_state_dict = get_pretrained_decoder(model.decoder, config, tokenizer.pad_token_id)

Loading weights:   0%|          | 0/318 [00:00<?, ?it/s]

[transformers] MBartForCausalLM LOAD REPORT from: hyunwoongko/asian-bart-ecjk
Key                                                       | Status     |  | 
----------------------------------------------------------+------------+--+-
model.encoder.layers.{0...11}.self_attn.q_proj.bias       | UNEXPECTED |  | 
model.encoder.layers.{0...11}.fc1.bias                    | UNEXPECTED |  | 
model.encoder.layers.{0...11}.self_attn.q_proj.weight     | UNEXPECTED |  | 
model.encoder.layers.{0...11}.self_attn.v_proj.weight     | UNEXPECTED |  | 
model.encoder.layers.{0...11}.fc2.weight                  | UNEXPECTED |  | 
model.encoder.layers.{0...11}.self_attn_layer_norm.bias   | UNEXPECTED |  | 
model.encoder.layers.{0...11}.final_layer_norm.bias       | UNEXPECTED |  | 
model.encoder.layers.{0...11}.self_attn.out_proj.bias     | UNEXPECTED |  | 
model.encoder.embed_tokens.weight                         | UNEXPECTED |  | 
model.encoder.layers.{0...11}.final_layer_norm.weight     | UNEXPECTED |  |

In [35]:
model.decoder.model.load_state_dict(decoder_state_dict)

<All keys matched successfully>

### Save Tensors

In [36]:
repo_dir = "./invoice-doc-to-text"
model.save_pretrained(repo_dir)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [37]:
from huggingface_hub import login, upload_folder
from datasets import load_dataset
from huggingface_hub import HfApi
import os
from dotenv import load_dotenv
from huggingface_hub import HfApi
load_dotenv(override=True)

login(token=os.environ["HF_TOKEN"])

repo_id = "ridwanFatur98/invoice-doc-to-text"

api = HfApi()

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [38]:
api.upload_file(
    path_or_fileobj=f"{repo_dir}/model.safetensors",
    path_in_repo="model.safetensors",
    repo_id=repo_id,
    repo_type="model",
)

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/ridwanFatur98/invoice-doc-to-text/commit/5a97c74bb5af9be897f6c8f9dbf02bbdfcd88ac7', commit_message='Upload model.safetensors with huggingface_hub', commit_description='', oid='5a97c74bb5af9be897f6c8f9dbf02bbdfcd88ac7', pr_url=None, repo_url=RepoUrl('https://huggingface.co/ridwanFatur98/invoice-doc-to-text', endpoint='https://huggingface.co', repo_type='model', repo_id='ridwanFatur98/invoice-doc-to-text'), pr_revision=None, pr_num=None)